# 05 — Business analysis

This notebook connects alert quality, analyst workload, false positives and rule tuning. Results are calculated from the generated database.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/generated/fraud_command_center.db')
alerts = pd.read_sql('SELECT * FROM alerts', conn)
transactions = pd.read_sql('SELECT * FROM transactions', conn)
review = alerts.merge(transactions[['transaction_id','is_fraud','amount']], on='transaction_id')
review['is_false_positive'] = ~review['is_fraud'].astype(bool)
review[['risk_level','is_fraud','is_false_positive']].groupby('risk_level').mean()

In [ ]:
workload = review.assign(analyst_minutes=8).groupby('alert_status')['analyst_minutes'].sum()
print(workload)
print('Synthetic false-positive rate:', review['is_false_positive'].mean())

## Recommendations

Tune rules using a precision-recall-workload balance. Do not optimize for minimum alert volume alone, and do not treat synthetic labels as proof of real-world fraud.